In [ ]:
import os

os.environ["PYKEOPS_VERBOSE"] = "0"
os.environ["KEOPS_VERBOSE"] = "0"

In [ ]:
import torch

# Force CUDA re-initialisation
if torch.cuda.is_available():
    torch.cuda.init()

print(torch.cuda.device_count())  # should now be > 0

In [ ]:
import sys
import os
import pickle
import logging
import warnings
import re
from typing import Dict, List
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.gridspec import GridSpec
import geopandas as gpd

import torch

import massbalancemachine as mbm

sys.path.append(os.path.join(os.getcwd(), '../../'))
from regions.TF_Europe.scripts.config_TF_Europe import *
from regions.TF_Europe.scripts.config_models import *
from regions.TF_Europe.scripts.dataset import *
from regions.TF_Europe.scripts.plotting import *
from regions.TF_Europe.scripts.models import *

warnings.filterwarnings('ignore')
%load_ext autoreload
%autoreload 2

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(message)s')

cfg = mbm.EuropeTFConfig()
mbm.utils.seed_all(cfg.seed)
mbm.utils.free_up_cuda()
mbm.plots.use_mbm_style()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(torch.cuda.is_available())
print(torch.cuda.device_count())

if torch.cuda.device_count() == 0:
    print("No GPU detected. Using CPU.")
    device = torch.device("cpu")
    
BASE_DIR = Path(cfg.dataPath) / path_cache / f"CROSS_REGION"
# make sure BASE_DIR exists
os.makedirs(BASE_DIR, exist_ok=True)
print(f"Base directory for data: {BASE_DIR}")

# Cross-regional modelling:

### Read stakes datasets:

In [ ]:
"""
Examples of loading data:
# Load Switzerland only
df = load_stakes(cfg, "CH")

# Load all Central Europe (SJM+CH+IT+AT when you add them)
df_ceu = load_stakes_for_rgi_region(cfg, "11")

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}"""

# Load all Europe regions configured
dfs = {rid: load_stakes_for_rgi_region(cfg, rid) for rid in RGI_REGIONS.keys()}

rgi_regions = dfs.keys()
num_measurements = {}
for rgi_region in rgi_regions:
    src_code_regions = dfs[rgi_region].SOURCE_CODE.unique()
    for src_code in src_code_regions:
        num_measurements[src_code] = len(
            dfs[rgi_region][dfs[rgi_region].SOURCE_CODE == src_code])
num_measurements

In [ ]:
region_colors = [
    "red", "blue", "green", "purple", "orange", "darkred", "cadetblue",
    "darkgreen", "darkpurple", "pink", "gray", "black"
]

# Central Asia
dfs, glacier_dict_ca = add_o2region_to_dfs(dfs, '13', cfg)
o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(
        sorted(v['O2Region'] for v in glacier_dict_ca.values()
               if v['O2Region'] is not None))
}
color_map = {
    g: o2_color_map[info['O2Region']]
    for g, info in glacier_dict_ca.items() if info['O2Region'] is not None
}
m = plot_stakes_folium(dfs['13'], color_map=color_map, tooltip_col='O2Region')
m

In [ ]:
# Alaska
dfs, glacier_dict_alaska = add_o2region_to_dfs(dfs,
                                               '01',
                                               cfg,
                                               deduplicate_glaciers=True)
o2_color_map = {
    r: region_colors[i % len(region_colors)]
    for i, r in enumerate(
        sorted(v['O2Region'] for v in glacier_dict_alaska.values()
               if v['O2Region'] is not None))
}
color_map = {
    g: o2_color_map[info['O2Region']]
    for g, info in glacier_dict_alaska.items() if info['O2Region'] is not None
}
m = plot_stakes_folium(dfs['01'], color_map=color_map, tooltip_col='O2Region')
m

### Monthly datasets:

In [ ]:
SOURCE_REGIONS = ['CH', 'NOR', 'ISL', "FR"]
TARGET_REGIONS = ['IT_AT', 'SJM', 'CENTRALASIA', 'ALA']

run_flag_by_code = {
    'ALA': False,
    'CENTRALASIA': False,
    'CH': False,
    'ISL': False,
    'NOR': False,
    'SJM': False,
    'FR': False,
    'IT_AT': False
}
# Compute monthly data once per code
monthly_cache = build_monthly_cache(
    cfg=cfg,
    dfs=dfs,
    paths_multi=paths_multi,
    vois_climate=VOIS_CLIMATE,
    vois_topographical=STATIC_COLS,
    region_codes=SOURCE_REGIONS + TARGET_REGIONS,
    run_flag_by_code=run_flag_by_code,
)

# Map O2Region into monthly_cache for CENTRALASIA
monthly_cache['CENTRALASIA']['data_monthly']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))
monthly_cache['CENTRALASIA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['CENTRALASIA']['data_monthly_aug']['GLACIER'].map({
        k:
        v['O2Region']
        for k, v in glacier_dict_ca.items()
    }))

# Map O2Region into monthly_cache for ALA
monthly_cache['ALA']['data_monthly']['O2Region'] = (
    monthly_cache['ALA']['data_monthly']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))
monthly_cache['ALA']['data_monthly_aug']['O2Region'] = (
    monthly_cache['ALA']['data_monthly_aug']['GLACIER'].map({
        k: v['O2Region']
        for k, v in glacier_dict_alaska.items()
    }))

# IT/AT split
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'IT_AT',
    subregions={
        'IT': IT_GLACIERS,
        'AT': AT_GLACIERS
    },
    drop_parent=True,
)

# Central Asia subregions (O2Region already mapped onto dfs['13'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'CENTRALASIA',
    subregions={
        'CA_12': {
            'o2region_col': 'O2Region',
            'values': ['1', '2']
        },
        'CA_3': {
            'o2region_col': 'O2Region',
            'values': ['3']
        },
        'CA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
    },
)

# Alaska subregions (O2Region already mapped onto dfs['01'])
monthly_cache = split_monthly_cache_by_glaciers(
    monthly_cache,
    'ALA',
    subregions={
        'ALA_2': {
            'o2region_col': 'O2Region',
            'values': ['2']
        },
        'ALA_4': {
            'o2region_col': 'O2Region',
            'values': ['4']
        },
        'ALA_6': {
            'o2region_col': 'O2Region',
            'values': ['6']
        },
    },
)

# With subregions
TARGET_REGIONS_SUB = [
    'IT', 'SJM', 'CENTRALASIA', 'ALA', 'CA_12', 'CA_3', 'CA_4', 'ALA_2',
    'ALA_4', 'ALA_6'
]

XREG_PAIRS = [(src,
               [r for r in SOURCE_REGIONS + TARGET_REGIONS_SUB if r != src])
              for src in SOURCE_REGIONS]

# Assemble train/test pairs from cache — no ERA5 reprocessing
res_xreg_by_source, split_info_by_source = prepare_xreg_pairs_from_cache(
    cfg=cfg,
    monthly_cache=monthly_cache,
    xreg_pairs=XREG_PAIRS,
)

In [ ]:
# --- inspect padding and FROM_DATE distribution per code ---
for code, entry in monthly_cache.items():
    print(f"\n{code}:")
    print(f"  head_pad: {entry['months_head_pad']}")
    print(f"  tail_pad: {entry['months_tail_pad']}")

    # check FROM_DATE months in the raw dfs
    for rid, df in dfs.items():
        if df is None or "SOURCE_CODE" not in df.columns:
            continue
        df_code = df[df["SOURCE_CODE"].str.upper() == code]
        if len(df_code) == 0:
            continue
        from_dt = pd.to_datetime(df_code["FROM_DATE"].astype(str),
                                 format="%Y%m%d")
        to_dt = pd.to_datetime(df_code["TO_DATE"].astype(str), format="%Y%m%d")
        print(
            f"  FROM_DATE month distribution:\n{from_dt.dt.month.value_counts().sort_index()}"
        )
        print(
            f"  TO_DATE month distribution:\n{to_dt.dt.month.value_counts().sort_index()}"
        )

### Domains shifts & feature overlap:

#### Domain shift:

In [ ]:
EXCLUDE_TARGETS = {}
res_all_xreg_by_source = {}

# The 4 individual targets we always want (each source excludes itself)
ALL_TARGETS = SOURCE_REGIONS + TARGET_REGIONS_SUB

for src_region in SOURCE_REGIONS:
    # Exclude the source region itself from targets
    target_codes = [t for t in ALL_TARGETS if t not in src_region]

    res_all_xreg = build_xreg_res_all(
        res_xreg=res_xreg_by_source[src_region],
        target_source_codes=target_codes,
        source_col="SOURCE_CODE",
        key_prefix=f"XREG_{src_region}_TO",
        ch_code=src_region,
        region_groups={},  # no group regions anymore
    )

    # Filter out excluded targets
    if EXCLUDE_TARGETS:
        res_all_xreg = {
            k: v
            for k, v in res_all_xreg.items()
            if not any(tgt in k for tgt in EXCLUDE_TARGETS)
        }

    res_all_xreg_by_source[src_region] = res_all_xreg
    print(f"{src_region} -> keys:", list(res_all_xreg.keys()))

In [ ]:
RECOMPUTE_SHIFTS = True

SHIFTS_CACHE = BASE_DIR / "domain_shifts_cache.pkl"

if RECOMPUTE_SHIFTS and SHIFTS_CACHE.exists():
    SHIFTS_CACHE.unlink()
    print(f"Deleted existing cache: {SHIFTS_CACHE}")

if SHIFTS_CACHE.exists():
    with open(SHIFTS_CACHE, "rb") as f:
        cache = pickle.load(f)
    scaler_m = cache["scaler_m"]
    scaler_s = cache["scaler_s"]
    blur_m = cache["blur_m"]
    blur_s = cache["blur_s"]
    blur_joint = cache["blur_joint"]
    bandwidths_m = cache["bandwidths_m"]
    bandwidths_s = cache["bandwidths_s"]
    all_shifts_by_source = cache["all_shifts_by_source"]
    print(
        f"Blurs from cache: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
    )
    print(f"Loaded shifts from cache: {SHIFTS_CACHE}")

else:
    scaler_m, scaler_s, scaler_joint = build_global_scalers_multi_source_simple(
        res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
    )

    blur_m, blur_s, blur_joint = estimate_global_bandwidths_simple(
        res_xreg_by_source,
        monthly_cols=MONTHLY_COLS,
        static_cols=STATIC_COLS,
        scaler_m=scaler_m,
        scaler_s=scaler_s,
        seed=cfg.seed,
    )
    bandwidths_m = [blur_m * 0.5, blur_m, blur_m * 2.0]
    bandwidths_s = [blur_s * 0.5, blur_s, blur_s * 2.0]
    print(
        f"Estimated blurs: blur_m={blur_m:.4f}, blur_s={blur_s:.4f}, blur_joint={blur_joint:.4f}"
    )

    all_shifts_by_source = {}
    for src_region in SOURCE_REGIONS:
        all_shifts = {}
        for key in tqdm(res_all_xreg_by_source[src_region], desc=src_region):
            shift = compute_domain_shift(
                df_src=res_all_xreg_by_source[src_region][key]["df_train"],
                df_tgt=res_all_xreg_by_source[src_region][key]["df_test"],
                monthly_cols=MONTHLY_COLS,
                static_cols=STATIC_COLS,
                scaler_m=scaler_m,
                scaler_s=scaler_s,
                scaler_joint=scaler_joint,
                blur_m=blur_m,
                blur_s=blur_s,
                blur_joint=blur_joint,
                bandwidths_m=bandwidths_m,
                bandwidths_s=bandwidths_s,
                seed=cfg.seed,
            )
            all_shifts[key] = shift
        all_shifts_by_source[src_region] = all_shifts

    with open(SHIFTS_CACHE, "wb") as f:
        pickle.dump(
            {
                "scaler_m": scaler_m,
                "scaler_s": scaler_s,
                "blur_m": blur_m,
                "blur_s": blur_s,
                "blur_joint": blur_joint,
                "bandwidths_m": bandwidths_m,
                "bandwidths_s": bandwidths_s,
                "all_shifts_by_source": all_shifts_by_source,
            }, f)
    print(f"Saved shifts to cache: {SHIFTS_CACHE}")

# Plot summary across regions, one figure per source
for src_region, all_shifts in all_shifts_by_source.items():
    print(f"\nDomain shift summary for source: {src_region}")
    fig = plot_domain_shift_across_regions(all_shifts, src_region=src_region)
    plt.show()

In [ ]:
for key in res_all_xreg:
    Num_measurements = res_all_xreg[key]['df_test']['ID'].nunique()
    print(key, 'Target region num_measurements:', Num_measurements)

In [ ]:
selected_cols = MONTHLY_COLS + [
    c for c in STATIC_COLS if c != "ELEVATION_DIFFERENCE"
] + ["POINT_BALANCE"]

glaciers_to_plot = {
    f"FR": {
        "df": res_all_xreg_by_source['CH']['XREG_CH_TO_FR']["df_test"],
        "color": COLORS['FR'],
    },
    f"CH ": {
        "df": res_all_xreg_by_source['CH']['XREG_CH_TO_NOR']["df_train"],
        "color": COLORS['CH'],
    },
    f"ISL": {
        "df": res_all_xreg_by_source['ISL']['XREG_ISL_TO_NOR']["df_train"],
        "color": COLORS['ISL'],
    },
    f"NOR": {
        "df": res_all_xreg_by_source['NOR']['XREG_NOR_TO_ISL']["df_train"],
        "color": COLORS['NOR'],
    },
}

plot_kde_pair(
    glaciers_to_plot=glaciers_to_plot,
    selected_cols=selected_cols,
    save_prefix="",
)

## ML models
### Model datasets:

In [ ]:
logs_cache_dir = BASE_DIR / "datasets"
outputs_xreg_by_source = {}

# NaN check on augmented dataframes before building LSTM datasets
print("--- NaN check on augmented dataframes ---")
for src_region, res_xreg in res_xreg_by_source.items():
    for split_name, df in [("df_train_aug", res_xreg["df_train_aug"]),
                           ("df_test_aug", res_xreg["df_test_aug"])]:
        feat_cols = [c for c in MONTHLY_COLS + STATIC_COLS if c in df.columns]
        n_nan_feat = df[feat_cols].isna().sum().sum()
        n_nan_tgt = df["POINT_BALANCE"].isna().sum(
        ) if "POINT_BALANCE" in df.columns else "N/A"
        print(
            f"  {src_region} {split_name}: feature NaNs={n_nan_feat}, target NaNs={n_nan_tgt}"
        )

for src_region in SOURCE_REGIONS:
    target_codes = [
        t for t in SOURCE_REGIONS + TARGET_REGIONS_SUB if t != src_region
    ]

    outputs_xreg_by_source[src_region] = build_or_load_lstm_all_xreg(
        cfg=cfg,
        res_xreg=res_xreg_by_source[src_region],
        MONTHLY_COLS=MONTHLY_COLS,
        STATIC_COLS=STATIC_COLS,
        cache_dir=logs_cache_dir / src_region,
        ch_code=src_region,
        target_source_codes=target_codes,
        region_groups={},  # no group regions anymore
    )
    print(f"{src_region} -> LSTM dataset keys:",
          list(outputs_xreg_by_source[src_region].keys()))

    # NaN/Inf check on the resulting datasets
    print(f"--- NaN/Inf check for {src_region} ---")
    for exp_key, assets in outputs_xreg_by_source[src_region].items():
        for ds_name in ["ds_train", "ds_test"]:
            ds = assets.get(ds_name)
            if ds is None:
                continue
            # Check directly on the underlying tensors (no iteration needed)
            checks = {"x_m": ds.Xm, "x_s": ds.Xs, "y": ds.y}
            any_issue = False
            for name, t in checks.items():
                n_nan = torch.isnan(t).sum().item()
                n_inf = torch.isinf(t).sum().item()
                if n_nan > 0 or n_inf > 0:
                    print(
                        f"  {exp_key} {ds_name} {name}: NaNs={n_nan}, Infs={n_inf}"
                    )
                    any_issue = True
            if not any_issue:
                print(f"  {exp_key} {ds_name}: OK")

### Train model:

#### LSTM:

In [ ]:
models_by_source_lstm = {}
infos_by_source_lstm = {}

TRAIN_FLAG = True  # set to True to retrain from scratch

for src_region in SOURCE_REGIONS:
    # Pick any key to get the train assets — they're the same across all targets
    any_key = next(iter(outputs_xreg_by_source[src_region]))
    train_assets = outputs_xreg_by_source[src_region][any_key]

    model, model_path, info = train_or_load_one_source_model(
        cfg=cfg,
        key=src_region,
        lstm_assets=train_assets,
        best_params=PARAMS_LSTM,
        device=device,
        models_dir=BASE_DIR / "models" / src_region,
        prefix=f"lstm_xreg",
        train_flag=TRAIN_FLAG,
        force_retrain=True,
        epochs=N_EPOCHS,
        date=MODEL_DATE,
        model_type="lstm",
    )

    models_by_source_lstm[src_region] = model
    infos_by_source_lstm[src_region] = {
        "model_path": model_path,
        **(info or {})
    }
    print(f"{src_region} -> model trained/loaded from {model_path}")

In [ ]:
df_metrics_by_source_lstm = {}
preds_by_source_lstm = {}
figs_by_source_lstm = {}

# Preferred display order for target regions — any not listed appear at the end
DISPLAY_ORDER = [
    'CH',
    'FR',
    'IT'
    'NOR',
    'ISL',
    'SJM',
    'ALA',
    'CENTRALASIA',
]

EXCLUDE_TARGETS = {'CENTRALASIA', 'ALA'}
for src_region in SOURCE_REGIONS:
    print(f"\nEvaluating source region: {src_region}")
    target_keys = [
        k for k in outputs_xreg_by_source[src_region].keys()
        if k.split("_TO_")[-1] not in EXCLUDE_TARGETS
    ]
    models_by_key = {
        key: models_by_source_lstm[src_region]
        for key in target_keys
    }

    # Derive custom_order from actual keys, sorted by preferred display order
    available_targets = [k.split("_TO_")[-1] for k in target_keys]
    custom_order = [t for t in DISPLAY_ORDER if t in available_targets]
    # append any targets not in DISPLAY_ORDER at the end
    custom_order += [t for t in available_targets if t not in custom_order]

    df_metrics, preds_by_key, figs_by_key, fig_grid = evaluate_all_models(
        cfg=cfg,
        models_by_key=models_by_key,
        lstm_assets_by_key=outputs_xreg_by_source[src_region],
        device=device,
        save_dir=f"figures/paperTF/eval_xreg/{src_region}",
        ax_xlim=(-16, 10),
        ax_ylim=(-16, 10),
        grid_shape=(4, 3),
        grid_figsize=(25, 12),
        domain_shifts=all_shifts_by_source[src_region],
        complement_key=f'XREG_{src_region}_TO_',
        custom_order=custom_order,
    )

    df_metrics_by_source_lstm[src_region] = df_metrics
    preds_by_source_lstm[src_region] = preds_by_key
    figs_by_source_lstm[src_region] = figs_by_key

#### Transformer:

In [ ]:
models_by_source_transformer = {}
infos_by_source_transformer = {}

TRAIN_FLAG = True

for src_region in SOURCE_REGIONS:
    any_key = next(iter(outputs_xreg_by_source[src_region]))
    train_assets = outputs_xreg_by_source[src_region][any_key]

    model, model_path, info = train_or_load_one_source_model(
        cfg=cfg,
        key=src_region,
        lstm_assets=train_assets,
        best_params=PARAMS_TRANSFORMER,
        device=device,
        models_dir=BASE_DIR / "models" / src_region,
        prefix=f"transformer_xreg",
        train_flag=TRAIN_FLAG,
        force_retrain=False,
        epochs=N_EPOCHS,
        date=MODEL_DATE,
        model_type="transformer",
    )

    models_by_source_transformer[src_region] = model
    infos_by_source_transformer[src_region] = {
        "model_path": model_path,
        **(info or {})
    }
    print(f"{src_region} -> model trained/loaded from {model_path}")

In [ ]:
df_metrics_by_source_transformer = {}
preds_by_source_transformer = {}
figs_by_source_transformer = {}

# Preferred display order for target regions — any not listed appear at the end
for src_region in SOURCE_REGIONS:
    print(f"\nEvaluating source region: {src_region}")
    target_keys = [
        k for k in outputs_xreg_by_source[src_region].keys()
        if k.split("_TO_")[-1] not in EXCLUDE_TARGETS
    ]
    models_by_key = {
        key: models_by_source_transformer[src_region]
        for key in target_keys
    }

    # Derive custom_order from actual keys, sorted by preferred display order
    available_targets = [k.split("_TO_")[-1] for k in target_keys]
    custom_order = [t for t in DISPLAY_ORDER if t in available_targets]
    # append any targets not in DISPLAY_ORDER at the end
    custom_order += [t for t in available_targets if t not in custom_order]

    df_metrics, preds_by_key, figs_by_key, fig_grid = evaluate_all_models(
        cfg=cfg,
        models_by_key=models_by_key,
        lstm_assets_by_key=outputs_xreg_by_source[src_region],
        device=device,
        save_dir=f"figures/eval_xreg/{src_region}",
        ax_xlim=(-16, 10),
        ax_ylim=(-16, 10),
        grid_shape=(4, 3),
        grid_figsize=(25, 12),
        domain_shifts=all_shifts_by_source[src_region],
        complement_key=f'XREG_{src_region}_TO_',
        custom_order=custom_order,
    )

    df_metrics_by_source_transformer[src_region] = df_metrics
    preds_by_source_transformer[src_region] = preds_by_key
    figs_by_source_transformer[src_region] = figs_by_key

### Comparison of models:

In [ ]:
src = "CH"
targets = ["FR", "ISL", "ALA_4", "CA_3"]  # adjust as needed

ncols = len(targets)
fig, axes = plt.subplots(
    2,
    ncols,
    figsize=(14, (90 * 2) / 25.4),
    sharex=True,
    sharey=True,
)

for row_i, (model_label, models_by_source) in enumerate([
    ("LSTM", models_by_source_lstm),
    ("Transformer", models_by_source_transformer),
]):
    model = models_by_source[
        src]  # one model trained on src, applied to all targets

    for col_i, tgt in enumerate(targets):
        ax = axes[row_i, col_i]
        key = f"XREG_{src}_TO_{tgt}"

        out, test_df_preds, created_fig, ax = evaluate_one_model(
            cfg=cfg,
            model=model,
            device=device,
            lstm_assets_for_key=outputs_xreg_by_source[src][key],
            ax=ax,
            ax_xlim=(-16, 10),
            ax_ylim=(-16, 10),
            title=f"{src} → {tgt}",
            legend_fontsize=NATURE_SPECS["font_min_pt"],
        )
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)

        panel_label = chr(ord('a') + row_i * ncols + col_i)
        ax.text(
            0.02,
            0.98,
            f"({panel_label})",
            transform=ax.transAxes,
            fontsize=NATURE_SPECS["font_max_pt"],
            fontweight="bold",
            va="top",
            ha="left",
        )

        leg = ax.get_legend()
        leg.remove()

        if col_i == 0:
            ax.set_ylabel(
                f"{model_label}\nmodeled PMB (m w.e.)",
                fontsize=NATURE_SPECS["font_max_pt"],
            )
        else:
            ax.set_ylabel("")

        if row_i == 0:
            ax.set_xlabel("")

fig.legend(
    handles=[
        mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
        mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.0),
    ncols=2,
    frameon=False,
    fontsize=NATURE_SPECS["font_max_pt"],
)
fig.suptitle(
    f"Zero-shot transfer — source: {src}  |  LSTM (top) vs Transformer (bottom)",
    fontsize=NATURE_SPECS["font_max_pt"] + 2,
    y=1.01,
)
fig.tight_layout(rect=[0, 0.05, 1, 0.98])

fig.savefig(f"figures/paperTF/zeroshot_{src}_lstm_vs_transformer.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()

In [ ]:
src = "CH"
targets = ["FR", "ISL", "ALA_4", "CA_3"]

TARGET_LABELS = {
    "FR": "France (FR)",
    "ISL": "Iceland (ISL)",
    "ALA_4": "Alaska 4",
    "CA_3": "Central Asia 3",
    **REGION_LABELS,
}

# Precompute per-target axis limits from test data
target_lims = {}
for tgt in targets:
    key = f"XREG_{src}_TO_{tgt}"
    assets = outputs_xreg_by_source[src][key]
    vals = assets["ds_test"].y.cpu().numpy().flatten()
    vals = vals[~np.isnan(vals)]
    pad = (vals.max() - vals.min()) * 0.3
    lim = (float(vals.min() - pad), float(vals.max() + pad))
    target_lims[tgt] = lim

ncols = len(targets)
fig, axes = plt.subplots(
    2,
    ncols,
    figsize=(4 * ncols, (90 * 2) / 25.4),
    sharex="col",
    sharey="col",
)

for row_i, (model_label, models_by_source) in enumerate([
    ("LSTM", models_by_source_lstm),
    ("Transformer", models_by_source_transformer),
]):
    model = models_by_source[src]

    for col_i, tgt in enumerate(targets):
        ax = axes[row_i, col_i]
        key = f"XREG_{src}_TO_{tgt}"
        lim = target_lims[tgt]

        out, test_df_preds, created_fig, ax = evaluate_one_model(
            cfg=cfg,
            model=model,
            device=device,
            lstm_assets_for_key=outputs_xreg_by_source[src][key],
            ax=ax,
            ax_xlim=lim,
            ax_ylim=lim,
            legend_fontsize=NATURE_SPECS["font_min_pt"],
        )
        apply_nature_style(ax, fontsize=NATURE_SPECS["font_max_pt"], box=True)
        leg = ax.get_legend()
        if leg:
            leg.remove()

        # subplot label
        row_letter = chr(ord('a') + row_i)
        panel_label = f"{row_letter}{col_i + 1}"
        ax.text(0.02,
                0.98,
                f"({panel_label})",
                transform=ax.transAxes,
                fontsize=NATURE_SPECS["font_max_pt"],
                fontweight="bold",
                va="top",
                ha="left")

        if row_i == 0:
            ax.set_title(TARGET_LABELS.get(tgt, tgt),
                         fontsize=NATURE_SPECS["font_max_pt"] + 1,
                         fontweight="bold")
        if col_i == 0:
            ax.set_ylabel(f"{model_label}\nModeled PMB (m w.e.)",
                          fontsize=NATURE_SPECS["font_max_pt"])
        else:
            ax.set_ylabel("")
        if row_i == 0:
            ax.set_xlabel("")

fig.legend(
    handles=[
        mpatches.Patch(color=mbm.plots.COLOR_ANNUAL, label="Annual"),
        mpatches.Patch(color=mbm.plots.COLOR_WINTER, label="Winter"),
    ],
    loc="lower center",
    bbox_to_anchor=(0.5, 0.0),
    ncols=2,
    frameon=False,
    fontsize=NATURE_SPECS["font_max_pt"],
)
fig.suptitle(
    f"Zero-shot transfer — source: {src}  |  LSTM (top) vs Transformer (bottom)",
    fontsize=NATURE_SPECS["font_max_pt"] + 2,
    y=1.01,
)
fig.tight_layout(rect=[0, 0.05, 1, 0.98])
fig.savefig(f"figures/paperTF/zeroshot_{src}_lstm_vs_transformer.png",
            dpi=NATURE_SPECS["dpi"],
            bbox_inches="tight")
plt.show()